# 리틀사이언스팟 - Option B 워크플로우 아키텍처

## 구현 요약
이 노트북은 리틀사이언스팟을 **워크플로우 아키텍처(Option B)** 로 구현한 예제입니다.

### 포함된 패턴
- **Prompt Chaining**: 기획 → 리서치 → 스크립트 → 검토 → 요약 순서로 진행
- **Parallelization**: 리서치 단계에서 3개의 워커가 병렬로 실행
- **Orchestrator-Workers**: 오케스트레이터가 과학 사실, 영어 단어, 부모 팁 워커에 작업을 분배
- **Conditional Edge**: 사용자 검토 결과에 따라 승인 또는 수정 경로로 분기
- **Multiple Tools**: `science_fact_tool`, `vocabulary_tool`, `parent_tip_tool` 사용

### 그래프 구조
```mermaid
graph TD
    A[부모 입력] --> B[plan_episode]
    B --> C[orchestrate_research]
    C --> D1[fetch_science_fact]
    C --> D2[fetch_vocabulary]
    C --> D3[fetch_parent_tip]
    D1 --> E[write_script]
    D2 --> E
    D3 --> E
    E --> F[review_script]
    F -->|approve| G[summarize_for_parent]
    F -->|revise| H[revise_script]
    H --> G
    G --> I[END]
```


## 상태 설계
- 사용자 입력: `age_group`, `topic`, `duration_minutes`, `tone`
- 검토 입력: `review_status`, `revision_request`
- 리서치 결과: `science_fact`, `target_words`, `parent_tip`
- 생성 결과: `plan`, `script`, `parent_summary`


In [ ]:
from typing import Literal, TypedDict

from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph


In [ ]:
class EpisodeState(TypedDict, total=False):
    age_group: str
    topic: str
    duration_minutes: int
    tone: str
    review_status: Literal["approve", "revise"]
    revision_request: str
    plan: str
    science_fact: str
    target_words: list[str]
    parent_tip: str
    script: str
    parent_summary: str


@tool
def science_fact_tool(topic: str) -> str:
    """Return a simple science fact for a children's audio script."""
    facts = {
        "the moon": "The moon does not make its own light. It reflects light from the sun, and it moves around Earth.",
        "rainbows": "A rainbow appears when sunlight passes through tiny drops of water and the light spreads into many colors.",
        "dinosaurs": "Dinosaurs lived a long time ago, and scientists learn about them by studying fossils in rocks.",
    }
    return facts.get(topic.lower(), f"{topic.title()} is a science topic we can explain with one simple fact and repeated English phrases.")


@tool
def vocabulary_tool(topic: str) -> list[str]:
    """Return simple English vocabulary words related to the topic."""
    words = {
        "the moon": ["moon", "light", "night"],
        "rainbows": ["rainbow", "color", "rain"],
        "dinosaurs": ["dinosaur", "bone", "fossil"],
    }
    return words.get(topic.lower(), ["science", "learn", "fun"])


@tool
def parent_tip_tool(age_group: str) -> str:
    """Return a parent coaching tip matched to the child's age group."""
    tips = {
        "4-5": "Pause after each short line and let the child repeat one keyword.",
        "6-8": "Ask the child to repeat the key phrase and explain one idea in their own words.",
        "9-10": "Ask one follow-up why-question after the episode to deepen understanding.",
    }
    return tips.get(age_group, "Repeat one key phrase together after listening.")


In [ ]:
def plan_episode(state: EpisodeState) -> EpisodeState:
    plan = (
        f"- Audience: {state['age_group']}\n"
        f"- Topic: {state['topic']}\n"
        f"- Length: about {state['duration_minutes']} minutes\n"
        f"- Tone: {state['tone']}\n"
        "- Characters: Curious Kid Leo, Dr. Spark\n"
        "- Goal: teach one science idea with easy English and repetition"
    )
    return {"plan": plan}


def orchestrate_research(state: EpisodeState) -> EpisodeState:
    return {"plan": f"{state['plan']}\n- Research workflow: fact, vocabulary, and parent tip in parallel"}


def fetch_science_fact(state: EpisodeState) -> EpisodeState:
    return {"science_fact": science_fact_tool.invoke({"topic": state["topic"]})}


def fetch_vocabulary(state: EpisodeState) -> EpisodeState:
    return {"target_words": vocabulary_tool.invoke({"topic": state["topic"]})}


def fetch_parent_tip(state: EpisodeState) -> EpisodeState:
    return {"parent_tip": parent_tip_tool.invoke({"age_group": state["age_group"]})}


def write_script(state: EpisodeState) -> EpisodeState:
    topic = state["topic"]
    fact = state["science_fact"]
    words = ", ".join(state["target_words"])
    script = f"""
[Leo] Dr. Spark, I want to learn about {topic}!

[Dr. Spark] Great question, Leo. Here is a simple idea: {fact}

[Leo] Wow! Can you say it again?

[Dr. Spark] Sure. Learning about {topic} is fun. Look up! Ask why! Science is fun!

[Leo] Look up! Ask why! Science is fun!

[Dr. Spark] Today's words are: {words}.
""".strip()
    return {"script": script}


def review_script(state: EpisodeState) -> EpisodeState:
    return {"review_status": state.get("review_status", "approve")}


def route_after_review(state: EpisodeState) -> str:
    return state["review_status"]


def revise_script(state: EpisodeState) -> EpisodeState:
    revision_request = state.get("revision_request", "Make the explanation even simpler for young children.")
    revised_script = (
        f"{state['script']}\n\n"
        f"[Dr. Spark] Parent request noted: {revision_request}\n"
        "[Dr. Spark] Let me say it in an even easier way. Science can be simple and fun!"
    )
    return {"script": revised_script, "review_status": "approve"}


def summarize_for_parent(state: EpisodeState) -> EpisodeState:
    summary = (
        f"Topic: {state['topic']}. "
        f"Core fact: {state['science_fact']} "
        f"Target words: {', '.join(state['target_words'])}. "
        f"Parent tip: {state['parent_tip']} "
        "Repeated expressions: 'Look up!', 'Ask why!', 'Science is fun!'"
    )
    return {"parent_summary": summary}


In [ ]:
graph_builder = StateGraph(EpisodeState)

graph_builder.add_node("plan_episode", plan_episode)
graph_builder.add_node("orchestrate_research", orchestrate_research)
graph_builder.add_node("fetch_science_fact", fetch_science_fact)
graph_builder.add_node("fetch_vocabulary", fetch_vocabulary)
graph_builder.add_node("fetch_parent_tip", fetch_parent_tip)
graph_builder.add_node("write_script", write_script)
graph_builder.add_node("review_script", review_script)
graph_builder.add_node("revise_script", revise_script)
graph_builder.add_node("summarize_for_parent", summarize_for_parent)

graph_builder.add_edge(START, "plan_episode")
graph_builder.add_edge("plan_episode", "orchestrate_research")
graph_builder.add_edge("orchestrate_research", "fetch_science_fact")
graph_builder.add_edge("orchestrate_research", "fetch_vocabulary")
graph_builder.add_edge("orchestrate_research", "fetch_parent_tip")
graph_builder.add_edge("fetch_science_fact", "write_script")
graph_builder.add_edge("fetch_vocabulary", "write_script")
graph_builder.add_edge("fetch_parent_tip", "write_script")
graph_builder.add_edge("write_script", "review_script")
graph_builder.add_conditional_edges(
    "review_script",
    route_after_review,
    {
        "approve": "summarize_for_parent",
        "revise": "revise_script",
    },
)
graph_builder.add_edge("revise_script", "summarize_for_parent")
graph_builder.add_edge("summarize_for_parent", END)

graph = graph_builder.compile()
graph


In [ ]:
approved_result = graph.invoke(
    {
        "age_group": "6-8",
        "topic": "the moon",
        "duration_minutes": 3,
        "tone": "playful",
        "review_status": "approve",
    }
)

approved_result


In [ ]:
revised_result = graph.invoke(
    {
        "age_group": "6-8",
        "topic": "rainbows",
        "duration_minutes": 3,
        "tone": "gentle",
        "review_status": "revise",
        "revision_request": "Use shorter sentences and more repetition.",
    }
)

revised_result


## 정리
- `plan_episode -> orchestrate_research -> write_script` 구간은 Prompt Chaining입니다.
- `fetch_science_fact`, `fetch_vocabulary`, `fetch_parent_tip`는 병렬 워커 노드입니다.
- `orchestrate_research`가 워커에게 작업을 분배하므로 Orchestrator-Workers 패턴입니다.
- `review_script` 이후 승인/수정 분기를 사용하므로 Conditional Edge가 구현되어 있습니다.
